In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')

def newest(pattern, label):
    paths = list(INPUT.rglob(pattern))
    if not paths:
        raise FileNotFoundError(f'Attach saved output containing {label}: {pattern}')
    return max(paths, key=lambda p: p.stat().st_mtime)

# Locate the Day 7 binary simulation artifacts.
binary_sources = []
for mp in INPUT.rglob('simulation_metadata.json'):
    if mp.parent.name != 'binary':
        continue
    try:
        meta = json.loads(mp.read_text(encoding='utf-8'))
    except Exception:
        continue
    if meta.get('family') == 'binary' and int(meta.get('parameter_draws_unique', 0)) >= 100_000:
        binary_sources.append((meta.get('created_utc', ''), mp.parent))
if not binary_sources:
    raise FileNotFoundError('Attach Day 7 Cosmic swan Bayesian binary simulations.')
_, BINARY_DIR = sorted(binary_sources, key=lambda item: item[0])[-1]
observations = np.load(BINARY_DIR / 'observations.npy', mmap_mode='r')
parameters = pd.read_csv(BINARY_DIR / 'parameters.csv')
time_days = np.asarray(np.load(BINARY_DIR / 'time_days.npy'), dtype=float)
if observations.ndim != 2 or observations.shape != (len(parameters), len(time_days)):
    raise ValueError('Day 7 observations, parameters, and time axis do not match.')

NUTS_PATH = newest('day9_11_conditional_nuisance_diagnostics.csv', 'Day 9.11 conditional NUTS diagnostics')
MAP_PATH = newest('day9_9_map_parameter_recovery.csv', 'Day 9.9 MAP parameter estimates')
nuts = pd.read_csv(NUTS_PATH)
map_params = pd.read_csv(MAP_PATH)
required_diag = {'simulation_row', 'status', 'period_fixed_profile_min_days'}
if not required_diag.issubset(nuts.columns):
    raise ValueError(f'Day 9.11 diagnostics missing columns: {sorted(required_diag - set(nuts.columns))}')
CASE_ROWS = sorted(nuts.loc[nuts.status.astype(str).isin(['needs_review', 'failed']), 'simulation_row'].astype(int).tolist())
if not CASE_ROWS:
    raise ValueError('Day 9.11 has no needs_review/failed rows to refine.')
if max(CASE_ROWS) >= len(parameters):
    raise ValueError('A selected case row is outside the Day 7 simulation table.')
PERIOD_FIXED = nuts.set_index('simulation_row')['period_fixed_profile_min_days'].astype(float).to_dict()
MAP_REQUIRED = {'baseline_flux', 'phase', 'depth_primary', 'depth_ratio', 'duration_fraction',
                'ellipsoidal_amplitude', 'reflection_amplitude', 'noise_sigma'}
MAP_SEEDS = {}
for row in CASE_ROWS:
    subset = map_params[map_params.simulation_row.astype(int).eq(row)]
    if subset.parameter.duplicated().any():
        raise ValueError(f'Duplicate Day 9.9 MAP parameter rows for simulation row {row}.')
    seeds = subset.set_index('parameter').map_estimate.astype(float).to_dict()
    missing = MAP_REQUIRED - set(seeds)
    if missing or any(not np.isfinite(seeds[name]) for name in MAP_REQUIRED if name in seeds):
        raise ValueError(f'Invalid or missing Day 9.9 seeds for row {row}: {sorted(missing)}')
    if not np.isfinite(PERIOD_FIXED[row]) or not 0.1 <= PERIOD_FIXED[row] <= 13.0:
        raise ValueError(f'Invalid fixed period for row {row}: {PERIOD_FIXED[row]}')
    MAP_SEEDS[row] = seeds

OUT = WORK / ('day9-12-conditional-map-refinement-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True, exist_ok=False)
print('[PASS] Day 7 binary simulations:', BINARY_DIR)
print('[PASS] Day 9.9 MAP seeds:', MAP_PATH.parent)
print('[PASS] Day 9.11 diagnostics:', NUTS_PATH.parent)
print('Cases to refine:', CASE_ROWS)
print('Fixed profile periods:', {r: PERIOD_FIXED[r] for r in CASE_ROWS})
print('Output:', OUT)


def smooth_clip_np(x, sharpness=500.0):
    return (np.logaddexp(0.0, sharpness*x) - np.logaddexp(0.0, sharpness*(x-1.0))) / sharpness


def predict_np(t, baseline, period, phase, depth, ratio, duration, ellip, reflection):
    orb = np.mod(t/period-phase+0.5, 1.0)-0.5
    sec = np.mod(orb, 1.0)-0.5
    half = max(float(duration)/2.0, 1e-6)
    ingress = max(half*0.18, 1e-6)
    primary = smooth_clip_np((half+ingress-np.abs(orb))/(2.0*ingress))
    secondary = smooth_clip_np((half+ingress-np.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary + ellip*np.cos(4*np.pi*orb) + reflection*np.cos(2*np.pi*orb)

In [ ]:
bounds = [(0.95, 1.05), (0.0, 1.0), (0.002, 0.45), (0.0, 1.0),
          (0.005, 0.12), (0.0, 0.02), (-0.01, 0.01), (np.log(1e-4), np.log(0.03))]
parameter_names = ['baseline_flux', 'phase', 'depth_primary', 'depth_ratio', 'duration_fraction',
                   'ellipsoidal_amplitude', 'reflection_amplitude', 'log_noise']
truth_parameter_names = ['baseline_flux', 'phase', 'depth_primary', 'depth_ratio', 'duration_fraction',
                         'ellipsoidal_amplitude', 'reflection_amplitude', 'noise_sigma']
case_rows, parameter_rows, plot_rows = [], [], []

for row in CASE_ROWS:
    started = time.time()
    period = float(PERIOD_FIXED[row])
    seed = MAP_SEEDS[row]
    t = np.asarray(time_days, dtype=float)
    y = np.asarray(observations[row], dtype=float)
    valid = np.isfinite(t) & np.isfinite(y)
    t, y = t[valid], y[valid]
    order = np.argsort(t)
    t, y = t[order], y[order]
    t = t - t.min()
    if len(t) < 100:
        raise ValueError(f'Too few valid observations for row {row}: {len(t)}')

    theta0 = np.array([seed['baseline_flux'], seed['phase'], seed['depth_primary'], seed['depth_ratio'],
                       seed['duration_fraction'], seed['ellipsoidal_amplitude'], seed['reflection_amplitude'],
                       np.log(seed['noise_sigma'])], dtype=float)
    theta0 = np.array([np.clip(v, lo+1e-8, hi-1e-8) for v, (lo, hi) in zip(theta0, bounds)])

    def nll(theta):
        pred = predict_np(t, theta[0], period, theta[1], theta[2], theta[3], theta[4], theta[5], theta[6])
        log_noise = theta[7]
        z = (y-pred)/np.exp(log_noise)
        return float(0.5*np.sum(z*z + 2.0*log_noise + np.log(2.0*np.pi)))

    starts = []
    starts.append(theta0.copy())
    phase_shift = theta0.copy(); phase_shift[1] = (phase_shift[1]+0.5) % 1.0; starts.append(phase_shift)
    short_event = theta0.copy(); short_event[4] = np.clip(theta0[4]*0.75, 0.005001, 0.119999); starts.append(short_event)
    long_event = theta0.copy(); long_event[4] = np.clip(theta0[4]*1.25, 0.005001, 0.119999); starts.append(long_event)
    fits = [minimize(nll, x0, method='L-BFGS-B', bounds=bounds,
                     options={'maxiter': 1500, 'ftol': 1e-10}) for x0 in starts]
    successful = [fit for fit in fits if fit.success and np.isfinite(fit.fun)]
    finite = [fit for fit in fits if np.isfinite(fit.fun)]
    if not finite:
        case_rows.append({'simulation_row': row, 'status': 'optimizer_failed', 'period_fixed_days': period,
                          'error': 'No finite objective from any starting point.', 'seconds': round(time.time()-started, 1)})
        continue
    best = min(successful or finite, key=lambda fit: fit.fun)
    theta = np.asarray(best.x, dtype=float)
    pred = predict_np(t, theta[0], period, theta[1], theta[2], theta[3], theta[4], theta[5], theta[6])
    residual = y-pred
    sigma = float(np.exp(theta[7]))
    z = residual/sigma
    lag1 = float(np.corrcoef(residual[:-1], residual[1:])[0, 1]) if len(residual) > 2 and np.std(residual) > 0 else float('nan')
    near_bounds = [parameter_names[i] for i, (value, (lo, hi)) in enumerate(zip(theta, bounds))
                   if min(value-lo, hi-value) <= 1e-5*max(1.0, abs(hi-lo))]
    truth = parameters.iloc[row]
    truth_values = {name: float(truth[name]) for name in ['phase', 'depth_primary', 'depth_ratio', 'duration_fraction',
                    'ellipsoidal_amplitude', 'reflection_amplitude', 'noise_sigma']}
    truth_values['baseline_flux'] = 1.0
    estimates = dict(zip(parameter_names, theta.tolist()))
    estimates['phase'] = float(theta[1] % 1.0)
    estimates['noise_sigma'] = sigma
    estimates.pop('log_noise')
    phase_error = float((estimates['phase']-truth_values['phase']+0.5) % 1.0-0.5)
    for name in truth_parameter_names:
        value = estimates[name]
        target = truth_values[name]
        error = phase_error if name == 'phase' else value-target
        parameter_rows.append({'simulation_row': row, 'parameter': name,
            'estimate': float(value), 'known_truth_evaluation_only': float(target),
            'error': float(error), 'absolute_error': float(abs(error)),
            'period_fixed_days': period, 'fit_scope': 'conditional_MLE_at_profile_period'})
    status = 'mle_fit_ok' if best.success else 'mle_fit_review'
    case_rows.append({'simulation_row': row, 'status': status, 'period_fixed_days': period,
        'optimizer_success': bool(best.success), 'optimizer_message': str(best.message),
        'successful_starts': len(successful), 'finite_starts': len(finite), 'negative_log_likelihood': float(best.fun),
        'baseline_flux': estimates['baseline_flux'], 'phase': estimates['phase'],
        'depth_primary': estimates['depth_primary'], 'depth_ratio': estimates['depth_ratio'],
        'duration_fraction': estimates['duration_fraction'], 'ellipsoidal_amplitude': estimates['ellipsoidal_amplitude'],
        'reflection_amplitude': estimates['reflection_amplitude'], 'noise_sigma': sigma,
        'residual_rms': float(np.sqrt(np.mean(residual**2))), 'residual_lag1_correlation': lag1,
        'max_abs_standardized_residual': float(np.max(np.abs(z))),
        'parameters_near_bounds': ','.join(near_bounds), 'seconds': round(time.time()-started, 1)})
    plot_rows.append((row, t.copy(), y.copy(), pred.copy(), period))
    pd.DataFrame(case_rows).to_csv(OUT/'day9_12_conditional_mle_diagnostics.csv', index=False)
    pd.DataFrame(parameter_rows).to_csv(OUT/'day9_12_conditional_mle_parameters.csv', index=False)
    print(f'Conditional MAP row {row}: {case_rows[-1]}', flush=True)

if plot_rows:
    ncols = 2
    nrows = int(np.ceil(len(plot_rows)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows), squeeze=False, constrained_layout=True)
    for ax, (row, t, y, pred, period) in zip(axes.flat, plot_rows):
        ax.scatter(t, y, s=5, alpha=0.3, label='synthetic observations')
        ax.plot(t, pred, color='crimson', lw=0.9, label='conditional MLE fit')
        ax.set_title(f'Row {row} | fixed profile period {period:.6f} d')
        ax.set_xlabel('Time (days)'); ax.set_ylabel('Flux'); ax.legend(fontsize=8)
    for ax in axes.flat[len(plot_rows):]: ax.set_visible(False)
    fig.savefig(OUT/'day9_12_conditional_mle_fits.png', dpi=140)
    plt.show()

summary = {'created_utc': datetime.now(timezone.utc).isoformat(),
    'day9_11_input': str(NUTS_PATH.parent), 'day9_9_map_input': str(MAP_PATH.parent),
    'refined_rows': CASE_ROWS, 'period_fixed': True,
    'fit_success_count': int(sum(row.get('optimizer_success', False) for row in case_rows)),
    'fit_review_count': int(sum(not row.get('optimizer_success', False) for row in case_rows)),
    'interpretation_limits': [
        'These are conditional maximum-likelihood refinements at fixed, data-selected profile periods.',
        'The fit diagnostics do not establish posterior convergence or interval calibration.',
        'Known truth is evaluation-only and is not used by the optimizer.',
        'A good synthetic fit does not establish performance on real stars.'],
    'outputs': ['day9_12_conditional_mle_diagnostics.csv', 'day9_12_conditional_mle_parameters.csv',
                'day9_12_conditional_mle_fits.png']}
(OUT/'day9_12_conditional_mle_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(pd.DataFrame(case_rows).to_string(index=False))
print('Saved conditional MAP parameters:', OUT/'day9_12_conditional_mle_parameters.csv')
print(json.dumps(summary, indent=2))